# Bước 1: ảnh thật → ảnh Pixar (FLUX.1 Kontext + LoRA 'zqx style')
Cần: GPU T4, Internet On, Kaggle Secret `HF_TOKEN` (đã bật cho notebook này), dataset ảnh thật, dataset chứa `trig100.pt`.
Kết quả: /kaggle/working/real2pixar/pixar/<tên>.png (khoảng 105 giây mỗi ảnh cho bước Pixar, thêm một lượt FLUX cho bước quay mặt thẳng).

In [ ]:
import os, glob
BASE = '/kaggle/temp'
os.makedirs(f'{BASE}/code', exist_ok=True)
os.environ['HF_HOME'] = f'{BASE}/hf_cache'
tok = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no Kaggle secret HF_TOKEN:', type(e).__name__)
if not tok:
    for f in glob.glob('/kaggle/input/**/hf_token.txt', recursive=True):  # cách dự phòng: dataset private có file hf_token.txt
        tok = open(f).read().strip()
assert tok, 'Không thấy HF_TOKEN: thêm Kaggle Secret HF_TOKEN (Add-ons -> Secrets) và bật cho notebook'
os.environ['HF_TOKEN'] = tok
import torch
assert torch.cuda.is_available(), 'Cần GPU: Session options -> Accelerator = GPU T4'
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers==4.57.1 diffusers==0.35.2 peft accelerate bitsandbytes sentencepiece protobuf
!pip uninstall -y -q torchao
!python -c "from diffusers import FluxKontextPipeline; print('ok')"

In [ ]:
open('/kaggle/temp/code/real2pixar_front.py', 'w', encoding='utf-8').write('"""Stage A+B of real2pixar: real photo -> front-facing photo (FLUX Kontext zero-shot) -> Pixar (FLUX Kontext + LoRA-100, constant prompt \'zqx style\').\n\n  python real2pixar_front.py --input <dir of <id>.png> --output <out dir> --ckpt trig100.pt [--limit N]\nBoth passes share one NF4 transformer: pass 1 without LoRA, then the LoRA state is loaded in place for pass 2. Resumable (skips existing pngs).\nWrites <out>/front/<id>.png, <out>/pixar/<id>.png, <out>/grid.jpg (original | front | pixar) and <out>/params.json.\n"""\nimport argparse, copy, gc, json, time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\n\nMODEL = "black-forest-labs/FLUX.1-Kontext-dev"\nTARGETS = ["to_q", "to_k", "to_v", "to_out.0", "add_q_proj", "add_k_proj", "add_v_proj", "to_add_out"]  # same as flux_kontext.py (rank 16 LoRA)\nFRONT = ("Rotate the head so the person faces the camera directly: a straight front-facing portrait, head upright, both eyes looking at the camera, "\n         "symmetric pose. Keep the same person: identity, face shape, hair, skin tone, expression, clothing and background.")\nPIXAR = "zqx style"\n\nap = argparse.ArgumentParser()\nap.add_argument("--input"); ap.add_argument("--output"); ap.add_argument("--ckpt")\nap.add_argument("--front_in", help="dir of already front-facing <id>.png (e.g. the Qwen output): skips pass 1 and runs only the Pixar pass on them")\nap.add_argument("--limit", type=int, default=0); ap.add_argument("--seed", type=int, default=42)\nap.add_argument("--res", type=int, default=512); ap.add_argument("--steps", type=int, default=28)\nap.add_argument("--guidance", type=float, default=2.5)\nargs = ap.parse_args()\nassert torch.cuda.is_available(), "CUDA GPU required"\ndev = "cuda"; dtype = torch.float16  # T4\nR = args.res\nout = Path(args.output)\nfor d in ("front", "pixar"):\n    (out / d).mkdir(parents=True, exist_ok=True)\nfiles = sorted(Path(args.input).glob("*.png"))\nfiles = files[: args.limit] if args.limit else files\nassert files, f"no png in {args.input}"\nfront_dir = Path(args.front_in) if args.front_in else out / "front"\nif args.front_in:\n    missing = [p.name for p in files if not (front_dir / p.name).exists()]\n    assert not missing, f"no front-facing image in {front_dir} for {missing[:5]}"\n\n\ndef free():\n    gc.collect(); torch.cuda.empty_cache()\n\n\nfrom diffusers import BitsAndBytesConfig, FluxKontextPipeline, FluxTransformer2DModel\nfrom peft import LoraConfig\nfrom peft.utils import set_peft_model_state_dict\n\npipe = FluxKontextPipeline.from_pretrained(MODEL, transformer=None, vae=None, torch_dtype=dtype).to(dev)\nemb = {}\nwith torch.no_grad():\n    for k, t in (("front", FRONT), ("pixar", PIXAR)):\n        pe, pooled, _ = pipe.encode_prompt(t, prompt_2=None, device=dev)\n        emb[k] = (pe.to(dtype), pooled.to(dtype))\ndel pipe; free()\ntr = FluxTransformer2DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype, quantization_config=BitsAndBytesConfig(\n    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype)).to(dev)\npipe = FluxKontextPipeline.from_pretrained(MODEL, transformer=tr, text_encoder=None, text_encoder_2=None, tokenizer=None,\n                                           tokenizer_2=None, torch_dtype=dtype).to(dev)\npipe.set_progress_bar_config(disable=True)\nvae32 = copy.deepcopy(pipe.vae).float()  # fp16 VAE encodes inside the pipeline, fp32 copy decodes\n\n\ndef edit(img, key):\n    pe, pooled = emb[key]\n    g = torch.Generator(dev).manual_seed(args.seed)\n    with torch.autocast("cuda", dtype=dtype):\n        lat = pipe(image=img, prompt_embeds=pe, pooled_prompt_embeds=pooled, height=R, width=R, guidance_scale=args.guidance,\n                   num_inference_steps=args.steps, generator=g, _auto_resize=False, max_area=R * R, output_type="latent").images\n    assert torch.isfinite(lat).all(), "non-finite latents (fp16 overflow)"\n    z = pipe._unpack_latents(lat.float(), R, R, pipe.vae_scale_factor) / vae32.config.scaling_factor + vae32.config.shift_factor\n    with torch.no_grad():\n        px = vae32.decode(z, return_dict=False)[0]\n    im = pipe.image_processor.postprocess(px, output_type="pil")[0]\n    assert np.asarray(im).max() > 0, "all-black output"\n    return im\n\n\ntimings = {"front": [], "pixar": []}\nfor i, p in enumerate(files, 1):  # pass 1: frontalise (base model, no LoRA)\n    if args.front_in or (out / "front" / p.name).exists():\n        continue\n    t0 = time.time()\n    edit(Image.open(p).convert("RGB").resize((R, R), Image.LANCZOS), "front").save(out / "front" / p.name)\n    timings["front"].append(time.time() - t0); print(f"front {p.name}: {timings[\'front\'][-1]:.0f}s ({i}/{len(files)})", flush=True)\n\ntr.requires_grad_(False)\ntr.add_adapter(LoraConfig(r=16, lora_alpha=16, init_lora_weights="gaussian", target_modules=TARGETS))\nset_peft_model_state_dict(tr, torch.load(args.ckpt, map_location="cpu")["state"])\ntr.eval()\nfor i, p in enumerate(files, 1):  # pass 2: Pixar with the zqx LoRA, on the frontalised image\n    if (out / "pixar" / p.name).exists():\n        continue\n    t0 = time.time()\n    edit(Image.open(front_dir / p.name).convert("RGB").resize((R, R), Image.LANCZOS), "pixar").save(out / "pixar" / p.name)\n    timings["pixar"].append(time.time() - t0); print(f"pixar {p.name}: {timings[\'pixar\'][-1]:.0f}s ({i}/{len(files)})", flush=True)\n\nsheet = Image.new("RGB", (256 * 3, 256 * len(files)), "white")\nfor i, p in enumerate(files):\n    for j, im in enumerate((Image.open(p).convert("RGB"), Image.open(front_dir / p.name), Image.open(out / "pixar" / p.name))):\n        sheet.paste(im.resize((256, 256)), (256 * j, 256 * i))\n    ImageDraw.Draw(sheet).text((4, 256 * i + 4), p.stem, fill=(255, 255, 0))\nsheet.save(out / "grid.jpg", quality=90)  # columns: original | front-facing | Pixar zqx\n(out / "params.json").write_text(json.dumps({"model": MODEL, "front_prompt": FRONT, "pixar_prompt": PIXAR, "ckpt": str(args.ckpt), "seed": args.seed,\n                                             "steps": args.steps, "guidance": args.guidance, "res": R,\n                                             "mean_seconds": {k: float(np.mean(v)) if v else 0.0 for k, v in timings.items()}}, indent=2))\nprint("done")\n')

In [ ]:
import subprocess, sys, shutil
from PIL import Image
CK = glob.glob('/kaggle/input/**/trig100.pt', recursive=True); assert CK, 'Không thấy trig100.pt: Add Data -> dataset checkpoint LoRA'
CK = CK[0]; print('ckpt:', CK)
IMGS = sorted(f for f in glob.glob('/kaggle/input/**/*', recursive=True) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
assert IMGS, 'Không thấy ảnh nào: Add Data -> dataset ảnh thật'
P = f'{BASE}/photos'; os.makedirs(P, exist_ok=True)
for f in IMGS:  # script đọc *.png, nên đổi hết sang png; tên file = mã kết quả
    Image.open(f).convert('RGB').save(os.path.join(P, os.path.splitext(os.path.basename(f))[0] + '.png'))
print(len(IMGS), 'ảnh')
r = subprocess.run([sys.executable, '-u', f'{BASE}/code/real2pixar_front.py', '--input', P, '--output', '/kaggle/working/real2pixar', '--ckpt', CK])
assert r.returncode == 0, f'real2pixar lỗi, mã {r.returncode}'
print(sorted(os.listdir('/kaggle/working/real2pixar/pixar')))